# 04 – Model Evaluation & Hyperparameter Tuning

Topics covered:
1. Train / Validation / Test split best practices
2. Cross-validation (k-Fold, Stratified)
3. Bias-Variance trade-off & Learning curves
4. Hyperparameter tuning: GridSearchCV, RandomizedSearchCV
5. Evaluating classifiers: precision, recall, F1, ROC-AUC
6. Evaluating regressors: MAE, RMSE, R²

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    cross_val_score, learning_curve,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score,
    ConfusionMatrixDisplay, confusion_matrix
)
from scipy.stats import randint, uniform

%matplotlib inline
sns.set_theme(style='whitegrid')
np.random.seed(42)

## 1. Cross-Validation

In [ ]:
bc = load_breast_cancer()
X, y = bc.data, bc.target

clf = RandomForestClassifier(n_estimators=100, random_state=42)

# k-Fold
kf_scores  = cross_val_score(clf, X, y, cv=KFold(n_splits=5, shuffle=True, random_state=42))
# Stratified k-Fold (recommended for classification)
skf_scores = cross_val_score(clf, X, y, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))

print(f'k-Fold  CV: {kf_scores.mean():.3f} ± {kf_scores.std():.3f}')
print(f'Strat   CV: {skf_scores.mean():.3f} ± {skf_scores.std():.3f}')

## 2. Learning Curves – Bias vs Variance

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    clf, X, y, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy'
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, 'o-', label='Train', color='steelblue')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2, color='steelblue')
plt.plot(train_sizes, val_mean, 's-', label='Validation', color='coral')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2, color='coral')
plt.xlabel('Training size'); plt.ylabel('Accuracy')
plt.title('Learning Curves'); plt.legend()
plt.tight_layout(); plt.show()

## 3. Hyperparameter Tuning

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# GridSearchCV – exhaustive
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [None, 5, 10],
    'min_samples_split': [2, 5]
}
grid_search = GridSearchCV(RandomForestClassifier(random_state=42),
                           param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_search.fit(X_tr, y_tr)
print('Best GridSearch params:', grid_search.best_params_)
print('Best GridSearch F1    :', grid_search.best_score_:.3f)

In [ ]:
# RandomizedSearchCV – faster for large spaces
param_dist = {
    'n_estimators':      randint(50, 300),
    'max_depth':         [None, 5, 10, 20],
    'min_samples_split': randint(2, 10),
    'max_features':      ['sqrt', 'log2', None]
}
rand_search = RandomizedSearchCV(RandomForestClassifier(random_state=42),
                                 param_dist, n_iter=20, cv=5,
                                 scoring='f1', random_state=42, n_jobs=-1)
rand_search.fit(X_tr, y_tr)
print('Best RandomSearch params:', rand_search.best_params_)
print('Best RandomSearch F1    :', rand_search.best_score_:.3f)

## 4. Full Classification Evaluation

In [ ]:
best_clf = grid_search.best_estimator_
y_pred = best_clf.predict(X_te)
y_prob = best_clf.predict_proba(X_te)[:, 1]

print(f'Accuracy : {accuracy_score(y_te, y_pred):.3f}')
print(f'Precision: {precision_score(y_te, y_pred):.3f}')
print(f'Recall   : {recall_score(y_te, y_pred):.3f}')
print(f'F1-score : {f1_score(y_te, y_pred):.3f}')
print(f'ROC-AUC  : {roc_auc_score(y_te, y_prob):.3f}')

# ROC Curve
fpr, tpr, _ = roc_curve(y_te, y_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'AUC={roc_auc_score(y_te, y_prob):.3f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Curve')
plt.legend(); plt.show()

## 5. Regression Evaluation

In [ ]:
housing = fetch_california_housing(as_frame=True)
X_r = housing.data
y_r = housing.target

X_r_tr, X_r_te, y_r_tr, y_r_te = train_test_split(X_r, y_r, test_size=0.2, random_state=42)

reg = Ridge(alpha=1.0)
reg.fit(StandardScaler().fit_transform(X_r_tr), y_r_tr)
y_r_pred = reg.predict(StandardScaler().fit_transform(X_r_te))

print(f'MAE : {mean_absolute_error(y_r_te, y_r_pred):.3f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_r_te, y_r_pred)):.3f}')
print(f'R²  : {r2_score(y_r_te, y_r_pred):.3f}')

## 6. Key Takeaways

| Concept | Recommendation |
|---------|----------------|
| CV strategy | Use `StratifiedKFold` for classification |
| Bias | High train error → underfitting → more data / complex model |
| Variance | Gap train vs val → overfitting → regularise / more data |
| GridSearchCV | Small search space; exhaustive |
| RandomizedSearchCV | Large search space; faster |
| Metric choice | Imbalanced → use F1 / AUC, not accuracy |

**Next:** `03_Advanced/01_Neural_Networks.ipynb`